# 8. Embedding-based regression

## Objective

This notebook evaluates the 384-dimensional text embeddings with a standardized Ridge regressor. The same shuffled 5-fold cross-validation protocol is used so that the result is comparable with the handcrafted linguistic baseline.

The comparison below reports the actual embedding RMSE and Pearson correlation alongside the previously recorded linguistic Ridge result. This determines whether dense semantic representations are useful for this dataset on their own.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr

PROJECT_ROOT = r"C:\Users\rudra\Desktop\SHL-Grammar-Scoring"

transcripts = pd.read_csv(
    PROJECT_ROOT + r"\outputs\train_transcripts.csv"
)

embeddings = np.load(
    PROJECT_ROOT + r"\outputs\text_embeddings.npy"
)

y = transcripts["label"].values

print("Embeddings:", embeddings.shape)
print("Labels:", y.shape)

In [ ]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=10.0))
])

embedding_pred = cross_val_predict(
    model,
    embeddings,
    y,
    cv=kf
)

embedding_rmse = np.sqrt(
    mean_squared_error(y, embedding_pred)
)

embedding_pearson = pearsonr(
    y,
    embedding_pred
)[0]

print("Text Embedding + Ridge")
print("RMSE:", embedding_rmse)
print("Pearson:", embedding_pearson)

In [ ]:
print("Mean baseline")
print("RMSE: 1.2382")

print("\nLinguistic + Ridge")
print("RMSE: 0.9909")
print("Pearson: 0.5997")

print("\nText Embedding + Ridge")
print("RMSE:", embedding_rmse)
print("Pearson:", embedding_pearson)

## Interpretation

The recorded embedding Ridge result is RMSE `1.1835530266082672` and Pearson `0.5199663744536653`. This is better than the mean baseline but weaker than the recorded handcrafted linguistic Ridge result (RMSE `0.9908950705951046`, Pearson `0.5997339184017166`), so embeddings alone were not selected as the primary representation.